# Full-measurement purification: fixed 40 cycles, 100 samples

Nx=20; Ny=30,36,42,48,54,60; hard walls x=5,15; alpha1=1, alpha2=30,
nshell=1, raster-y, perfect correction, complex128. The entire physical layer
starts maximally mixed. meas_slab_only=False: measure both the slab and exterior,
without Born-conditioned exterior preparation. Requires A100 40-GB-class.

At every cycle t=6,...,40, save each sample's full occupation spectrum, raw
modular gap, finite-time gap divided by 2t, and all complex128 eigenvectors with
$10^{-9}<\nu<1-10^{-9}$. This includes zero-rate modes, not just the smallest-gap mode.
Vectors use the full spatial basis, with counts and spectrum indices for padding.
Gaps are computed sample by sample, not from the ensemble-averaged spectrum.

600 trajectories in 18 execution batches. 4,200 five-sample/cycle result pairs.
A rolling covariance/RNG checkpoint is published before spectral extraction at
every cycle. Completed spectral shards are retained and skipped on restart.
No covariance histories, entropy contours, bootstrap, or old-data migration.
Never run two writers on this output folder.

Saving modes at 35 cycles requires many more eigendecompositions than endpoint-only.
The uncompressed worst case is about 1.15 TB of eigenvectors; actual output depends
on the number of mixed modes. No A100 runtime estimate has been measured for this
new workload. Set MAX_NEW_EXECUTION_BATCHES=1 to measure the first Ny=60 batch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # use 1 for the first Ny=60 batch
RUN_ANALYSIS = False  # requires all 4,200 cycle result shards
# Complete configuration: scientific changes require a new revision.
CONFIG = {'sampling_revision': 'full_measurement_gap_nx20_ny30-60_s100_t40_cycles6-40_v1',
 'root_seed': 2026092828,
 'Nx': 20,
 'Ny_values': [30, 36, 42, 48, 54, 60],
 'samples_per_case': 100,
 'cycles': 40,
 'execution_batch_size_by_Ny': {'30': 100,
                                '36': 50,
                                '42': 50,
                                '48': 25,
                                '54': 25,
                                '60': 20},
 'result_shard_size': 5,
 'segment_cycles': 1,
 'DW': True,
 'dw_truncation': True,
 'meas_slab_only': False,
 'triv_region_local_mode': False,
 'alpha_1': 1.0,
 'alpha_2': 30.0,
 'nshell': 1,
 'trial_orbitals': 'X',
 'filling_fraction': 0.5,
 'init_mode': 'maxmix',
 'n_a': 0.5,
 'sequence': 'raster_y',
 'perfect_correction': True,
 'postselect': False,
 'state_representation': 'covariance',
 'dtype': 'complex128',
 'device': 'cuda:0',
 'backend': 'local',
 'cap_tolerance': 1e-09,
 'initial_purity_tolerance': 0.50000001,
 'gpu_allocator_limit_bytes': 35000000000,
 'spectrum_cycles': [6,
                     7,
                     8,
                     9,
                     10,
                     11,
                     12,
                     13,
                     14,
                     15,
                     16,
                     17,
                     18,
                     19,
                     20,
                     21,
                     22,
                     23,
                     24,
                     25,
                     26,
                     27,
                     28,
                     29,
                     30,
                     31,
                     32,
                     33,
                     34,
                     35,
                     36,
                     37,
                     38,
                     39,
                     40],
 'observer_sample_chunk': 1,
 'covariance_spectral_clip': False,
 'cycle_finite_mode_vectors': True,
 'finite_mode_basis': 'full_centered_covariance_eigenvectors'}
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/28_full_measurement_purification_gap_t40')
LOCAL_BUNDLE_DIR = Path('/content/28_full_measurement_purification_gap_t40')
SCRATCH_ROOT = Path('/content/full_measurement_gap_t40_scratch')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']


## Stage locally and run/resume

The largest geometry runs first. The runner prints checkpoint status, measured cycle/checkpoint times and GPU memory. A failed write leaves the task incomplete; each cycle checkpoint survives until its spectral shards verify; only then does evolution advance.

In [ ]:
import codecs
import json
import shutil
import subprocess
import sys

LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'endpoint_spectrum.py', 'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py', 'analyze_campaign.py']:
    destination = LOCAL_BUNDLE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, destination)
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))

def run_streamed(command):
    print('[launch]', ' '.join(command), flush=True)
    decoder = codecs.getincrementaldecoder('utf-8')('replace')
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0) as process:
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True)); sys.stdout.flush()
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)

command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
           '--config', str(config_path), '--output-root', str(OUTPUT_ROOT),
           '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    command.extend(['--max-new-execution-batches', str(MAX_NEW_EXECUTION_BATCHES)])
run_streamed(command)


## Optional cycle-resolved gap analysis

Mean of sample-resolved gaps with ordinary SEM; raw modular gap and normalized Lyapunov half gap are plotted separately. No fitted exponent.

In [ ]:
if RUN_ANALYSIS and not REPORT_ONLY:
    run_streamed([sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'analyze_campaign.py'),
                  '--output-root', str(OUTPUT_ROOT)])
else:
    print('Analysis skipped.')


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
